# Session 1 — Python preliminaries & eye-tracking data landscape
**90 minutes** · For participants new to Python

### What you will leave with
1. Confidence running notebook cells and reading simple tables.
2. A map of **which file answers which question**.
3. Names of the main Python libraries used in eye-tracking analysis.
4. Your first **summary table + bar chart** from real workshop data.

We keep tables small (a few rows) and charts simple (one idea per figure).


## Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Readable plots for projection
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else Path("workshop")
sys.path.insert(0, str(WORKSHOP_DIR))
from analysis.paths import data_path, stimuli_path
print('OK — data folder:', data_path('food_decision_making').exists())

## 1. Python in 10 minutes (no prior experience)

Think of a notebook cell like a calculator that can also hold tables.


In [ ]:
# Numbers and text
n_participants = 4
course = "NBML eye-tracking"
print(course, "| participants:", n_participants)

# A list (ordered collection)
foods = ["cake", "pizza", "ice-cream"]
print("first food:", foods[0])
print("how many foods:", len(foods))

In [ ]:
# A table = DataFrame (like an Excel sheet)
demo = pd.DataFrame({
    "AOI": ["food-pic", "buy", "not-buy"],
    "dwell_ms": [1200, 450, 300],
})
demo

### Presentation output — tiny bar chart
One chart, three bars. This is the style we want all session: **clear, not dense**.


In [ ]:
fig, ax = plt.subplots()
ax.bar(demo["AOI"], demo["dwell_ms"], color="#2a6f6f")
ax.set_ylabel("Dwell (ms)")
ax.set_title("Toy example — dwell by AOI")
plt.tight_layout()
plt.show()

## 2. Libraries you will hear about this week

| Library | Role in this course |
|---------|---------------------|
| **pandas** | Load CSV/TSV/XLSX, filter, group, summarize |
| **numpy** | Numeric arrays (velocity, pupil series) |
| **matplotlib** | Figures for slides |
| **scipy** / **pingouin** / **statsmodels** | Tests & models (Session 4) |
| **Pillow** | Open stimulus images under heatmaps (Session 2) |

Optional names to recognize later: neurokit2 (GSR), vendor SDKs, PyGaze.


## 3. Workshop data map

| Folder | Best for |
|--------|----------|
| `Data/food_decision_making/` | Sample-level gaze, mouse, AOI hits, pupil |
| `Data/tobii_gsr_demo/` | Aggregated GSR, SCR, AOI metrics, clicks |
| `Data/pupil_labs_recording/` | Wearable fixations & blinks |
| `Stimuli/Decision Making/` | Images for overlays |


## 4. Load food metrics (event table) and summarize safely

In [ ]:
metrics = pd.read_csv(
    data_path("food_decision_making", "Food_Decision_Making_Metrics.tsv"),
    sep="\t",
)
print("rows, cols:", metrics.shape)
metrics.head(5)

### Metric check before plotting
`Duration` in this Tobii metrics export is the **event duration**.
We only keep rows that look like fixations and have an AOI label.


In [ ]:
print("Event_type values:")
print(metrics["Event_type"].value_counts(dropna=False).head(8))

fix = metrics.copy()
# Keep fixation-like rows if the column uses that vocabulary
if "Event_type" in fix.columns:
    # Tobii often uses 'Fixation' — be explicit
    mask = fix["Event_type"].astype(str).str.contains("Fixation", case=False, na=False)
    if mask.any():
        fix = fix.loc[mask].copy()

fix = fix.dropna(subset=["AOI", "Duration"]).copy()
fix["Duration"] = pd.to_numeric(fix["Duration"], errors="coerce")
fix = fix.dropna(subset=["Duration"])

# Summarize: few AOIs only (top by count) so the table stays readable
top_aois = fix["AOI"].value_counts().head(6).index
summary = (
    fix[fix["AOI"].isin(top_aois)]
    .groupby("AOI", as_index=False)
    .agg(
        n_fixations=("Duration", "size"),
        mean_duration=("Duration", "mean"),
        total_dwell=("Duration", "sum"),
    )
    .sort_values("total_dwell", ascending=False)
)
# Round for slides
summary["mean_duration"] = summary["mean_duration"].round(1)
summary["total_dwell"] = summary["total_dwell"].round(1)
summary

In [ ]:
fig, ax = plt.subplots()
ax.barh(summary["AOI"], summary["total_dwell"], color="#345995")
ax.set_xlabel("Total dwell (same units as Duration column)")
ax.set_title("Food metrics — total dwell by AOI (top 6)")
plt.tight_layout()
plt.show()

## 5. Peek at sample-level teaching CSV (one recording)

In [ ]:
sample = pd.read_csv(data_path("food_decision_making", "Food_Decision_Making_Teaching_Sample.csv"))
# Compact overview table for presentation
overview = pd.DataFrame({
    "n_rows": [len(sample)],
    "n_eye_rows": [(sample["Sensor"] == "Eye Tracker").sum()],
    "n_mouse_rows": [(sample["Sensor"] == "Mouse").sum()],
    "n_stimuli": [sample["Presented Stimulus name"].nunique(dropna=True)],
})
overview

In [ ]:
stim_counts = (
    sample.loc[sample["Sensor"] == "Eye Tracker", "Presented Stimulus name"]
    .value_counts()
    .head(8)
    .rename_axis("stimulus")
    .reset_index(name="n_gaze_samples")
)
stim_counts

In [ ]:
fig, ax = plt.subplots()
ax.bar(stim_counts["stimulus"], stim_counts["n_gaze_samples"], color="#6b4c9a")
ax.set_ylabel("Gaze samples")
ax.set_title("Teaching recording — gaze samples by stimulus")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Pupil Labs vs Tobii (one small table each)

In [ ]:
pupil_fix = pd.read_csv(data_path("pupil_labs_recording", "fixations.csv"))
pupil_blinks = pd.read_csv(data_path("pupil_labs_recording", "blinks.csv"))
compare = pd.DataFrame([
    {"source": "Pupil Labs fixations.csv", "n_events": len(pupil_fix), "duration_col": "duration", "mean_duration": round(pupil_fix["duration"].mean(), 3)},
    {"source": "Pupil Labs blinks.csv", "n_events": len(pupil_blinks), "duration_col": "duration", "mean_duration": round(pupil_blinks["duration"].mean(), 3)},
])
compare

## 7. Practice (10 min)
1. From `summary`, which AOI has the largest total dwell?
2. Why might `total_dwell` use the same units as `Duration` (not always milliseconds)?
3. Exit ticket: which file will you open first for I-VT tomorrow — metrics TSV or teaching CSV — and why?
